# HTRC SF 1945–1980 — BERTopic pipeline

Runs in the capsule in secure mode over `/data/sf_corpus/<htid>/<page>.txt` and `metadata_august2026.csv`. Edit the configuration cell, run the preflight, then run top to bottom.


## 1. Configuration — edit before running


In [ ]:
import os
from pathlib import Path

# Capsule facts this notebook is built around:
#   secure mode = NO network. A from_pretrained() that reaches for the Hub does not fail, it HANGS. Force offline.
#   no GPU, no AVX: torch is a CUDA build running on CPU; encoding is the slow step and scales with threads.
#   ~50 vCPUs (dc6), ~19 GB free disk, /media/secure_volume only exists in secure mode.
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
TORCH_THREADS = None            # None = torch default (all cores); set e.g. 24 if the box is shared with word2vec runs

DRY_RUN = os.environ.get("BERTOPIC_DRY_RUN", "0") == "1"   # local smoke test on public-domain text

if DRY_RUN:
    ROOT = Path(os.environ.get("BERTOPIC_DRY_ROOT", "./dryrun")).resolve()
    TEXT_DIR     = ROOT / "sf_corpus"
    METADATA_CSV = ROOT / "metadata.csv"
    OUT_DIR      = ROOT / "out_bertopic"
    WORK_DIR     = ROOT / "work_bertopic"
    MODEL_PATH   = Path("sentence-transformers/all-MiniLM-L6-v2")   # from the local HF cache
else:
    TEXT_DIR     = Path("/data/sf_corpus")                           # per-HTID folders, per-page .txt
    METADATA_CSV = Path("/home/dcuser/metadata_august2026.csv")
    OUT_DIR      = Path("/media/secure_volume/out_bertopic")         # release-safe, survives the session
    WORK_DIR     = Path("/media/secure_volume/work_bertopic")        # never exported
    MODEL_PATH   = Path.home() / "models" / "minilm"                 # staged in maintenance mode

YEAR_CORRECTIONS_CSV = None   # e.g. Path("/home/dcuser/year_corrections.csv") with htid,year (original printing)

ID_COL, YEAR_COL, SOURCE_COL = "htid", "year", "source"
ERA_LABELS   = ["era_a", "era_b", "era_c"]        # same cutoffs as SF_word2vec_eras.ipynb
YEAR_CUTOFFS = [1962, 1972]                       # pre-Silent Spring | to Clean Water Act | after

# --- OCR cleaning (page level) ---
RUNNING_HEAD_MIN_PAGES = 3      # a short line recurring on >= this many pages AND >= RUNNING_HEAD_FRAC of pages
RUNNING_HEAD_FRAC      = 0.05   # ... is a running head / page furniture, not prose
RUNNING_HEAD_MAX_CHARS = 60
DICT_MIN_VOLS          = 10     # a word counts as real if it appears in >= this many volumes
PAGE_MIN_DICT_RATE     = 0.55   # drop pages whose share of real words is below this; None = keep every page

# --- Chunking (arm A, MiniLM, max_seq 256) ---
CHUNK_WORDS   = 165   # measured: 1.3749 subword tokens/word on this OCR -> 165 words ≈ 227 pieces, under the 254 budget
OVERLAP_WORDS = 25
MIN_CHUNK_WORDS = 40

# --- Sampling for the fit ---
SAMPLE_PER_VOL = 60     # evenly spaced chunks per volume for the fit sample
BALANCE_ERAS   = True   # downsample each era's fit chunks to the smallest era (mirrors subsample_to_match)
SAMPLE_SEED    = 1

# --- Embedding ---
BATCH_SIZE      = 64
CHECKPOINT_EVERY = 200   # batches between embedding checkpoints to WORK_DIR

# --- Assigning the chunks that were not in the fit sample (§10) ---
# At 2,881 volumes this is ~1.8M more chunks to encode on a CPU with no AVX — the long pole of the whole run,
# and the one step whose cost is optional. The fit sample already gives every volume SAMPLE_PER_VOL points.
#   "all"   -> encode + assign every remaining chunk (resumable per volume; expect many hours)
#   "none"  -> skip; volume-level tables use the fit sample only
#   int N   -> assign at most N further evenly spaced chunks per volume
ASSIGN_MODE = 120

# --- BERTopic ---
UMAP_COMPONENTS     = 5
UMAP_NEIGHBORS      = 15
HDBSCAN_MIN_CLUSTER = 110          # ~110 for this chunk volume; 50 was tuned for a far smaller corpus
MIN_DF              = 2            # ⚠ BERTopic fits the vectorizer on ONE document PER TOPIC, so min_df counts topics, not chunks:
                                   #   a term must appear in >= MIN_DF topics. 5 would need >= 5 topics to exist and drops any term
                                   #   specific to fewer than five topics. 2 is the usual BERTopic setting.
N_TOPICS_REDUCE     = None         # e.g. 60 -> a logged reduce_topics() step after the raw fit; None = keep raw
TOP_N_WORDS         = 30
MMR_DIVERSITY       = 0.3
SEEDS               = [1, 2, 3]    # UMAP seeds for the stability check (first = base fit)
FIX_BASE_SEED       = False        # True pins UMAP's random_state for the base fit (single-core, slow)

EXTRA_STOPWORDS = ["said", "would", "could", "should", "also", "one", "like", "back", "get", "got",
                   "mr", "mrs", "sir", "yes", "oh", "well", "still", "though", "even", "much", "way",
                   "went", "came", "come", "go", "going", "know", "knew", "thought", "think", "see",
                   "saw", "looked", "look", "made", "make", "take", "took", "put", "let", "say", "tell",
                   "told", "asked", "ask", "man", "men", "eyes", "face", "hand", "hands", "head", "time"]

# Her lexicon, verbatim from SF_word2vec_eras.ipynb — topics are scored against these groups in §9.
ENV_WORD_GROUPS = {
    "landscape_baseline": ["river", "creek", "stream", "water", "forest", "nature", "wilderness", "jungle",
                            "ocean", "landscape", "levee", "dam", "reservoir", "estuary", "wetland",
                            "marsh", "watershed"],
    "ecology_concept": ["ecology", "ecosystem", "environment", "biosphere", "habitat", "balance", "cycle"],
    "contamination": ["radiation", "radioactive", "fallout", "contamination", "waste", "smog", "fumes",
                       "chemical", "pesticide", "exhaust", "toxic", "polluted", "pollution"],
    "waste_infrastructure": ["sewer", "sewage", "drainage", "effluent", "runoff", "wastewater",
                              "cesspool", "sludge", "septic", "cistern", "culvert", "plumbing"],
    "population_scarcity": ["overpopulation", "population", "famine", "scarcity", "starvation", "resource", "drought"],
    "energy": ["oil", "fuel", "energy", "coal", "nuclear", "power"],
    "disaster_collapse": ["wasteland", "extinction", "collapse", "barren", "dying", "decay", "catastrophe"],
    "climate_weather": ["climate", "weather", "warming", "greenhouse", "atmosphere", "temperature",
                         "flood", "flooding", "storm", "hurricane", "ice", "glacier", "carbon", "ozone"],
    "space_earth_framing": ["earth", "homeworld", "colony", "frontier", "terraform", "alien"],
}

OUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
(WORK_DIR / "chunks").mkdir(exist_ok=True)
if not DRY_RUN:
    for p in (OUT_DIR, WORK_DIR):
        assert str(p).startswith("/media/secure_volume"), f"{p} would not survive the session"
print("DRY_RUN" if DRY_RUN else "CAPSULE", "| text:", TEXT_DIR, "| out:", OUT_DIR, "| work:", WORK_DIR)

### 1b. Preflight — run this before anything else

Checks secure mode, the staged encoder, `punkt`, package versions, threads and disk; fails with the fix.


In [ ]:
# PREFLIGHT — everything this run needs, checked before anything expensive. Fails loudly, with the fix.
import shutil, sys, importlib, platform, time

problems, notes = [], []
notes.append(f"python {platform.python_version()}  cpus {os.cpu_count()}")

# 1. secure mode / persistence
if not DRY_RUN:
    if not Path("/media/secure_volume").exists():
        problems.append("/media/secure_volume is absent -> the capsule is in MAINTENANCE mode. Switch to secure mode first; "
                        "in maintenance mode nothing written here survives, and the text is not mounted.")
    if not TEXT_DIR.exists():
        problems.append(f"{TEXT_DIR} not found -> text not mounted / wrong TEXT_DIR")
    if not METADATA_CSV.exists():
        problems.append(f"{METADATA_CSV} not found -> stage it from maintenance mode (scp) or fix the path")

# 2. model staged (no network to fetch it)
if not DRY_RUN and not Path(MODEL_PATH).expanduser().exists():
    problems.append(f"{MODEL_PATH} missing -> download in MAINTENANCE mode: "
                    "python -c \"from sentence_transformers import SentenceTransformer as S; S('sentence-transformers/all-MiniLM-L6-v2').save('~/models/minilm')\"")

# 3. NLTK punkt (sentence splitting). Regex fallback exists, but punkt is what the eras notebook uses.
try:
    import nltk; nltk.data.find("tokenizers/punkt"); notes.append("punkt: staged")
except Exception:
    notes.append("punkt: NOT staged -> regex sentence splitter will be used. To stage (maintenance mode): "
                 "python -c \"import nltk; nltk.download('punkt'); nltk.download('punkt_tab')\" or copy an nltk_data/tokenizers dir to ~/nltk_data")

# 4. package versions vs the ones measured in the capsule's BERT env
expected = {"bertopic": "0.17.4", "sentence_transformers": "2.6.1", "umap": "0.5.12", "numpy": "1.24.3"}
for mod, ver in expected.items():
    try:
        v = importlib.import_module(mod).__version__
        notes.append(f"{mod} {v}" + ("" if v == ver else f"  (measured in capsule: {ver} — different env/kernel?)"))
    except Exception as e:
        problems.append(f"{mod} not importable in this kernel ({e}) -> pick the kernel of the env that has the BERTopic stack (was: env 'BERT')")
try:
    import torch
    if TORCH_THREADS: torch.set_num_threads(TORCH_THREADS)
    notes.append(f"torch {torch.__version__} threads {torch.get_num_threads()} cuda {torch.cuda.is_available()}")
except Exception as e:
    problems.append(f"torch not importable ({e})")

# 5. disk on the secure volume: chunk text (~6 bytes/word) + fit embeddings + model
free_gb = shutil.disk_usage(WORK_DIR).free / 1e9
notes.append(f"free on {WORK_DIR}: {free_gb:.1f} GB")
if free_gb < 5:
    problems.append(f"only {free_gb:.1f} GB free at {WORK_DIR}; chunk text for 2,881 volumes is ~1.7 GB plus embeddings — clear space or point WORK_DIR elsewhere on the secure volume")

print("\n".join("  " + n for n in notes))
if problems:
    print("\nPREFLIGHT FAILED:\n" + "\n".join("  ✗ " + p for p in problems))
    raise SystemExit("fix the above before running the rest")
print("\npreflight OK")

## 2. Load metadata → era


In [ ]:
import pandas as pd

meta = pd.read_csv(METADATA_CSV, dtype=str)
meta[ID_COL] = meta[ID_COL].str.strip()
if SOURCE_COL not in meta.columns:
    meta[SOURCE_COL] = "unknown"

if YEAR_CORRECTIONS_CSV and Path(YEAR_CORRECTIONS_CSV).exists():
    corr = pd.read_csv(YEAR_CORRECTIONS_CSV, dtype=str)
    corr[ID_COL] = corr[ID_COL].str.strip()
    fix = dict(zip(corr[ID_COL], corr[YEAR_COL]))
    n_fix = meta[ID_COL].isin(fix).sum()
    meta[YEAR_COL] = meta.apply(lambda r: fix.get(r[ID_COL], r[YEAR_COL]), axis=1)
    print(f"applied {n_fix} original-printing year corrections")

bad = meta[~meta[YEAR_COL].astype(str).str.fullmatch(r"\d{4}")]
assert bad.empty, f"{len(bad)} rows without a 4-digit year — fix the metadata, do not bin these:\n{bad.head()}"
meta[YEAR_COL] = meta[YEAR_COL].astype(int)

# duplicate htids = multi-work volumes listed once per work (the metadata file has 11). One row per volume.
dups = meta[meta[ID_COL].duplicated(keep=False)]
if not dups.empty:
    print(f"{dups[ID_COL].nunique()} htids appear more than once (multi-work volumes); keeping the first row of each")
    meta = meta.drop_duplicates(subset=[ID_COL], keep="first")

def assign_eras_by_year(df, id_col=ID_COL, year_col=YEAR_COL, cutoffs=None, labels=None):
    labels = labels or ERA_LABELS
    cutoffs = cutoffs or YEAR_CUTOFFS
    assert len(labels) == len(cutoffs) + 1
    bounds = [-float("inf")] + list(cutoffs) + [float("inf")]
    era = pd.Series(index=df.index, dtype=object)
    for label, lo, hi in zip(labels, bounds[:-1], bounds[1:]):
        era[(df[year_col] >= lo) & (df[year_col] < hi)] = label
    return era

meta["era"] = assign_eras_by_year(meta)
meta = meta.set_index(ID_COL, drop=False)
meta_lookup = meta.to_dict("index")
print(f"{len(meta):,} volumes")
print(meta.groupby("era")[YEAR_COL].agg(["count", "min", "max"]))
print(meta[SOURCE_COL].value_counts().to_dict())

## 3. Discover volumes — per-HTID folder, per-page files


In [ ]:
def discover_volumes(base_dir, ids=None):
    """Return {htid: [page paths in filename order]} for htid folders that have pages."""
    base_dir = Path(base_dir)
    vols = {}
    for d in sorted(p for p in base_dir.iterdir() if p.is_dir() and not p.name.startswith(".")):
        if ids is not None and d.name not in ids:
            continue
        pages = sorted(d.glob("*.txt"))
        if pages:
            vols[d.name] = pages
    return vols

volumes = discover_volumes(TEXT_DIR, ids=set(meta.index))
unmatched_dirs = [d.name for d in TEXT_DIR.iterdir() if d.is_dir() and d.name not in meta.index]
missing_text = [h for h in meta.index if h not in volumes]
print(f"{len(volumes):,} volumes with text and metadata")
print(f"{len(unmatched_dirs):,} folders without metadata (skipped), e.g. {unmatched_dirs[:3]}")
print(f"{len(missing_text):,} metadata rows without text, e.g. {missing_text[:3]}")

## 4. Clean pages (OCR)


In [ ]:
# ---- OCR cleaning, shared by SF_word2vec_eras.ipynb and htrc_bertopic_pipeline.ipynb. Keep both copies identical. ----
import os, re
from collections import Counter
from multiprocessing import get_context
from pathlib import Path

_norm_ws = re.compile(r"\s+")
_only_number = re.compile(r"^\s*[\divxlcIVXLC]+\s*$")
_hyphen_break = re.compile(r"(\w)-\s*\n\s*(\w)")
_word = re.compile(r"[A-Za-z']+")


def discover_volumes(base_dir, ids=None):
    """{htid: [page paths in filename order]} for every htid folder that has pages. Loose files are skipped."""
    vols = {}
    for d in sorted(p for p in Path(base_dir).iterdir() if p.is_dir() and not p.name.startswith(".")):
        if ids is not None and d.name not in ids:
            continue
        pages = sorted(d.glob("*.txt"))
        if pages:
            vols[d.name] = pages
    return vols


def _norm_line(line):
    return _norm_ws.sub(" ", re.sub(r"\d+", "", line)).strip().lower()


def _volume_vocab(page_paths):
    words = set()
    for p in page_paths:
        words.update(w.lower() for w in _word.findall(p.read_text(encoding="utf-8", errors="replace")) if len(w) > 1)
    return words


def build_dictionary(volumes, min_vols, procs=None):
    """Words that occur in at least min_vols volumes: the corpus's own dictionary, so OCR garbage scores low."""
    df = Counter()
    with get_context("fork").Pool(procs or min(16, os.cpu_count() or 1)) as pool:
        for vocab in pool.imap_unordered(_volume_vocab, list(volumes.values()), chunksize=8):
            df.update(vocab)
    return {w for w, c in df.items() if c >= min_vols}


def page_dict_rate(text, dictionary):
    words = [w.lower() for w in _word.findall(text) if len(w) > 1]
    return sum(1 for w in words if w in dictionary) / len(words) if words else 0.0


def clean_volume(page_paths, dictionary=None):
    """Return (text, stats): running heads and page-number lines dropped, low-dictionary pages dropped
    when PAGE_MIN_DICT_RATE is set, pages joined and hyphenated line breaks rejoined."""
    pages = [p.read_text(encoding="utf-8", errors="replace") for p in page_paths]
    line_pages = Counter()
    per_page_lines = []
    for text in pages:
        lines = text.split("\n")
        per_page_lines.append(lines)
        line_pages.update({_norm_line(l) for l in lines if 0 < len(l.strip()) <= RUNNING_HEAD_MAX_CHARS})
    thresh = max(RUNNING_HEAD_MIN_PAGES, int(RUNNING_HEAD_FRAC * len(pages)))
    heads = {l for l, c in line_pages.items() if c >= thresh and l}
    dropped_lines = dropped_pages = 0
    kept = []
    for lines in per_page_lines:
        out = []
        for l in lines:
            if (len(l.strip()) <= RUNNING_HEAD_MAX_CHARS and _norm_line(l) in heads) or _only_number.match(l):
                dropped_lines += 1
                continue
            out.append(l)
        page_text = "\n".join(out)
        if dictionary is not None and PAGE_MIN_DICT_RATE and page_dict_rate(page_text, dictionary) < PAGE_MIN_DICT_RATE:
            dropped_pages += 1
            continue
        kept.append(page_text)
    text, n_dehyph = _hyphen_break.subn(r"\1\2", "\n".join(kept))
    stats = {"pages": len(pages), "running_heads": len(heads), "dropped_lines": dropped_lines,
             "dropped_pages": dropped_pages, "dehyphenated": n_dehyph, "words": len(text.split())}
    return text, stats


In [ ]:
dictionary = build_dictionary(volumes, DICT_MIN_VOLS) if PAGE_MIN_DICT_RATE else None
if dictionary is not None:
    print(f"dictionary: {len(dictionary):,} words appear in >= {DICT_MIN_VOLS} of {len(volumes):,} volumes")

# smoke check on one volume
_h = next(iter(volumes))
_t, _s = clean_volume(volumes[_h], dictionary)
print(_h, _s)


## 5. Chunk — sentence-packed, verified against the live tokenizer


In [ ]:
import time
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(str(MODEL_PATH))

# bertopic 0.17.4 imports `StaticEmbedding`, missing from the capsule's sentence-transformers 2.6.1.
# Wrapping the encoder in BaseEmbedder keeps select_backend() from importing that module.
from bertopic.backend import BaseEmbedder

class STEmbedder(BaseEmbedder):
    def __init__(self, model):
        super().__init__()
        self.embedding_model = model
    def embed(self, documents, verbose=False):
        return self.embedding_model.encode(documents, show_progress_bar=verbose,
                                           batch_size=BATCH_SIZE, convert_to_numpy=True)

bt_embedder = STEmbedder(embedder)
tokenizer = embedder.tokenizer
TOKEN_BUDGET = embedder.max_seq_length - 2
print("embedder max_seq", embedder.max_seq_length, "-> budget", TOKEN_BUDGET)

# 30-second benchmark so the runtime is known BEFORE committing the session to it (no AVX: expect tens of chunks/s)
_probe = ["word " * CHUNK_WORDS] * 256
_t = time.time(); embedder.encode(_probe, batch_size=BATCH_SIZE, show_progress_bar=False); _rate = len(_probe) / (time.time() - _t)
print(f"encode rate ≈ {_rate:.0f} chunks/s on this box")

try:
    import nltk
    nltk.data.find("tokenizers/punkt")
    from nltk.tokenize import sent_tokenize
    _split = sent_tokenize
    print("sentence splitter: nltk punkt")
except Exception:
    _sent_re = re.compile(r"(?<=[.!?])\s+(?=[A-Z\"'])")
    _split = lambda t: _sent_re.split(t)
    print("sentence splitter: regex fallback (punkt not staged)")

def pack_chunks(text, size=CHUNK_WORDS, overlap=OVERLAP_WORDS, min_words=MIN_CHUNK_WORDS):
    sents = [s.strip() for s in _split(text.replace("\n", " ")) if s.strip()]
    chunks, cur, cur_n = [], [], 0
    for s in sents:
        n = len(s.split())
        if n > size:                       # a single monster 'sentence' (OCR run-on): hard-split it
            words = s.split()
            for i in range(0, len(words), size):
                chunks.append(" ".join(words[i:i + size]))
            cur, cur_n = [], 0
            continue
        if cur_n + n > size and cur:
            chunks.append(" ".join(cur))
            # carry the tail as overlap
            tail, tail_n = [], 0
            for t in reversed(cur):
                tn = len(t.split())
                if tail_n + tn > overlap:
                    break
                tail.insert(0, t); tail_n += tn
            cur, cur_n = tail, tail_n
        cur.append(s); cur_n += n
    if cur_n >= min_words:
        chunks.append(" ".join(cur))
    return [c for c in chunks if len(c.split()) >= min_words]

def enforce_budget(chunks):
    oversized = 0
    for ch in chunks:
        n = len(tokenizer.tokenize(ch))
        if n <= TOKEN_BUDGET:
            yield ch; continue
        oversized += 1
        parts = int(n / TOKEN_BUDGET) + 1
        words = ch.split()
        step = max(MIN_CHUNK_WORDS, len(words) // parts)
        for i in range(0, len(words), step):
            piece = " ".join(words[i:i + step])
            if len(piece.split()) >= MIN_CHUNK_WORDS:
                yield piece
    enforce_budget.oversized = getattr(enforce_budget, "oversized", 0) + oversized

enforce_budget.oversized = 0
chunk_index = []           # dicts without text: htid, chunk, era, year, source, n_words
clean_stats = {}
t0 = time.time()
for k, (htid, pages) in enumerate(volumes.items(), 1):
    text, stats = clean_volume(pages, dictionary)
    clean_stats[htid] = stats
    m = meta_lookup[htid]
    with (WORK_DIR / "chunks" / f"{htid}.jsonl").open("w") as fh:
        for j, ch in enumerate(enforce_budget(pack_chunks(text))):
            fh.write(json.dumps({"c": j, "t": ch}) + "\n")
            chunk_index.append({"htid": htid, "chunk": j, "era": m["era"], "year": m[YEAR_COL],
                                "source": m[SOURCE_COL], "n_words": len(ch.split())})
    if k % 50 == 0 or k == len(volumes):
        print(f"  {k}/{len(volumes)} volumes, {len(chunk_index):,} chunks, {time.time() - t0:.0f}s")

chunk_df = pd.DataFrame(chunk_index)
chunk_df.to_csv(WORK_DIR / "chunk_index.csv", index=False)
json.dump(clean_stats, (WORK_DIR / "clean_stats.json").open("w"))
print(f"{len(chunk_df):,} chunks; {enforce_budget.oversized} needed splitting past the budget")
print(chunk_df.groupby('era').size())

## 6. Sample for the fit — per volume, era-balanced


In [ ]:
import numpy as np
rng = np.random.default_rng(SAMPLE_SEED)

fit_rows = []
for htid, grp in chunk_df.groupby("htid", sort=False):
    n = len(grp)
    k = min(SAMPLE_PER_VOL, n)
    pick = np.unique(np.linspace(0, n - 1, k).round().astype(int))
    fit_rows.extend(grp.index[pick].tolist())
fit_mask = chunk_df.index.isin(fit_rows)

if BALANCE_ERAS:
    counts = chunk_df[fit_mask].groupby("era").size()
    target = int(counts.min())
    keep = []
    for era, grp in chunk_df[fit_mask].groupby("era"):
        idx = grp.index.to_numpy()
        keep.extend(rng.choice(idx, size=min(target, len(idx)), replace=False).tolist())
    fit_mask = chunk_df.index.isin(keep)

chunk_df["in_fit"] = fit_mask
print("fit sample:", int(fit_mask.sum()), "chunks;", chunk_df[fit_mask].groupby("era").size().to_dict())
print("to assign later:", int((~fit_mask).sum()), f"(before ASSIGN_MODE={ASSIGN_MODE!r})")
print(f"fit-sample encode ≈ {int(fit_mask.sum()) / max(_rate, 1e-9) / 60:.0f} min at the benchmarked rate")

def read_chunks(htid):
    with (WORK_DIR / "chunks" / f"{htid}.jsonl").open() as fh:
        return {json.loads(l)["c"]: json.loads(l)["t"] for l in fh}

fit_df = chunk_df[fit_mask].reset_index(drop=True)
fit_texts = []
for htid, grp in fit_df.groupby("htid", sort=False):
    d = read_chunks(htid)
    fit_texts.extend([(i, d[c]) for i, c in zip(grp.index, grp["chunk"])])
fit_texts = [t for _, t in sorted(fit_texts)]
assert len(fit_texts) == len(fit_df)

## 7. Embed the fit sample — checkpointed to `WORK_DIR`


In [ ]:
emb_path = WORK_DIR / "embeddings_fit.npy"
if emb_path.exists() and np.load(emb_path, mmap_mode="r").shape[0] == len(fit_texts):
    fit_emb = np.load(emb_path)
    print("resumed", fit_emb.shape)
else:
    parts = []
    t0 = time.time()
    for b in range(0, len(fit_texts), BATCH_SIZE * CHECKPOINT_EVERY):
        block = fit_texts[b:b + BATCH_SIZE * CHECKPOINT_EVERY]
        parts.append(embedder.encode(block, batch_size=BATCH_SIZE, show_progress_bar=False, convert_to_numpy=True))
        np.save(WORK_DIR / "embeddings_fit.partial.npy", np.vstack(parts))
        print(f"  {min(b + len(block), len(fit_texts)):,}/{len(fit_texts):,}  {time.time() - t0:.0f}s")
    fit_emb = np.vstack(parts).astype(np.float32)
    np.save(emb_path, fit_emb)
    (WORK_DIR / "embeddings_fit.partial.npy").unlink(missing_ok=True)
print(fit_emb.shape)

## 8. Fit BERTopic


In [ ]:
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import MaximalMarginalRelevance
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from umap import UMAP
from hdbscan import HDBSCAN

STOPWORDS = sorted(set(ENGLISH_STOP_WORDS) | set(EXTRA_STOPWORDS))
TOKEN_PATTERN = r"(?u)\b[a-z]+(?:'[a-z]+)?\b"      # SF_word2vec_eras TOKEN_RE, as a vectorizer pattern

def make_model(seed=None):
    vectorizer = CountVectorizer(stop_words=STOPWORDS, min_df=MIN_DF, ngram_range=(1, 2),
                                 token_pattern=TOKEN_PATTERN, lowercase=True)
    ctfidf = ClassTfidfTransformer(reduce_frequent_words=True)
    umap_model = UMAP(n_neighbors=UMAP_NEIGHBORS, n_components=UMAP_COMPONENTS, min_dist=0.0,
                      metric="cosine", low_memory=False, random_state=seed)
    hdb = HDBSCAN(min_cluster_size=HDBSCAN_MIN_CLUSTER, metric="euclidean",
                  cluster_selection_method="eom", prediction_data=True, core_dist_n_jobs=-1)
    return BERTopic(embedding_model=bt_embedder, umap_model=umap_model, hdbscan_model=hdb,
                    vectorizer_model=vectorizer, ctfidf_model=ctfidf,
                    representation_model=MaximalMarginalRelevance(diversity=MMR_DIVERSITY),
                    top_n_words=TOP_N_WORDS, nr_topics=None, calculate_probabilities=False, verbose=True)

t0 = time.time()
topic_model = make_model(seed=SEEDS[0] if FIX_BASE_SEED else None)
topics, _ = topic_model.fit_transform(fit_texts, fit_emb)
n_topics_raw = len(set(topics)) - (1 if -1 in topics else 0)
print(f"raw fit: {n_topics_raw} topics, {sum(t == -1 for t in topics):,} outliers, {time.time() - t0:.0f}s")

if N_TOPICS_REDUCE:
    topic_model.reduce_topics(fit_texts, nr_topics=N_TOPICS_REDUCE)
    topics = topic_model.topics_
    print(f"reduced to {len(set(topics)) - (1 if -1 in topics else 0)} topics")

fit_df["topic"] = topics
# model stays in WORK_DIR: its saved form can carry representative documents (raw text) — verify before ever releasing
topic_model.save(str(WORK_DIR / "bertopic_model"), serialization="safetensors", save_ctfidf=True, save_embedding_model=False)

## 9. Inspect topics — `Representative_Docs` stripped — and score them against the env lexicon

`get_topic_info()` carries raw chunk text; it is dropped before anything is written.


In [ ]:
info = topic_model.get_topic_info()
info = info.drop(columns=[c for c in info.columns if "Representative" in c], errors="ignore")
info.to_csv(OUT_DIR / "topic_info.csv", index=False)
print(info[info.Topic != -1].head(30).to_string())

rows = []
for tid in sorted(set(topics)):
    for rank, (term, w) in enumerate(topic_model.get_topic(tid) or []):
        rows.append((tid, rank, term, round(float(w), 6)))
terms_df = pd.DataFrame(rows, columns=["topic", "rank", "term", "weight"])
terms_df.to_csv(OUT_DIR / "topic_terms.csv", index=False)

env_rows = []
for tid, grp in terms_df.groupby("topic"):
    row = {"topic": tid}
    for gname, words in ENV_WORD_GROUPS.items():
        ws = set(words)
        row[gname] = round(float(sum(w for term, w in zip(grp.term, grp.weight)
                                     if any(part in ws for part in term.split()))), 6)
    row["env_total"] = round(sum(v for k, v in row.items() if k != "topic"), 6)
    env_rows.append(row)
env_df = pd.DataFrame(env_rows).sort_values("env_total", ascending=False)
env_df.to_csv(OUT_DIR / "topic_env_scores.csv", index=False)
print(env_df.head(15).to_string(index=False))

## 10. Assign the remaining chunks in batches


In [ ]:
assign_path = WORK_DIR / "assignments.csv"
done = set()
if assign_path.exists():
    done = set(pd.read_csv(assign_path, usecols=["htid"])["htid"].unique())
    print("resuming; volumes already assigned:", len(done))
else:
    assign_path.write_text("htid,chunk,topic\n")

vec_sum = {h: np.zeros(fit_emb.shape[1], dtype=np.float64) for h in volumes}
vec_n = {h: 0 for h in volumes}
# fold the fit sample in first
for h, e in zip(fit_df["htid"], fit_emb):
    vec_sum[h] += e; vec_n[h] += 1

rest_df = chunk_df[~chunk_df.in_fit]
if ASSIGN_MODE == "none":
    rest_df = rest_df.iloc[0:0]
elif isinstance(ASSIGN_MODE, int):
    keep = []
    for htid, grp in rest_df.groupby("htid", sort=False):
        n = len(grp); kk = min(ASSIGN_MODE, n)
        keep.extend(grp.index[np.unique(np.linspace(0, n - 1, kk).round().astype(int))].tolist())
    rest_df = rest_df.loc[keep]
est_h = len(rest_df) / max(_rate, 1e-9) / 3600
print(f"ASSIGN_MODE={ASSIGN_MODE!r}: {len(rest_df):,} chunks to encode ≈ {est_h:.1f} h at the benchmarked rate (resumable per volume)")
topic_model.verbose = False          # per-volume transform() would otherwise log four lines per book
t0 = time.time()
with assign_path.open("a") as fh:
    for k, (htid, grp) in enumerate(rest_df.groupby("htid", sort=False), 1):
        if htid in done:
            continue
        d = read_chunks(htid)
        texts = [d[c] for c in grp["chunk"]]
        if not texts:
            continue
        emb = embedder.encode(texts, batch_size=BATCH_SIZE, show_progress_bar=False, convert_to_numpy=True)
        tp, _ = topic_model.transform(texts, emb)
        vec_sum[htid] += emb.sum(axis=0); vec_n[htid] += len(texts)
        for c, t in zip(grp["chunk"], tp):
            fh.write(f"{htid},{c},{t}\n")
        if k % 25 == 0:
            fh.flush()
            print(f"  {k}/{rest_df['htid'].nunique()} volumes  {time.time() - t0:.0f}s")

assigned = pd.read_csv(assign_path)
all_df = chunk_df.merge(assigned, on=["htid", "chunk"], how="left")
all_df = all_df[all_df.in_fit | all_df.topic.notna()].copy()     # chunks skipped by ASSIGN_MODE carry no topic and drop out here
all_df.loc[all_df.in_fit, "topic"] = all_df.loc[all_df.in_fit].merge(
    fit_df[["htid", "chunk", "topic"]], on=["htid", "chunk"], how="left", suffixes=("", "_fit"))["topic_fit"].values
all_df["topic"] = all_df["topic"].astype(int)
# Index-only (no text), so release-SAFE — but ~40 bytes/chunk: ~80 MB at 2M chunks, far over the 1 MB review cap.
# It lives in WORK_DIR; the volume-level tables in §11–12 are the exportable form of the same information.
all_df.drop(columns=["n_words"]).to_csv(WORK_DIR / "chunk_topic_assignments.csv", index=False)
print(f"{len(all_df):,} of {len(chunk_df):,} chunks carry a topic ({int(all_df.in_fit.sum()):,} fit + {int((~all_df.in_fit).sum()):,} assigned); outlier share {(all_df.topic == -1).mean():.3f}")

## 11. Diachronic — eras and years


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

clean = all_df[all_df.topic != -1]
era_tot = clean.groupby("era").size()
by_era = clean.groupby(["era", "topic"]).size().reset_index(name="chunks")
by_era["proportion"] = by_era.apply(lambda r: r.chunks / era_tot[r.era], axis=1)
by_era.to_csv(OUT_DIR / "topic_by_era.csv", index=False)
wide = by_era.pivot(index="topic", columns="era", values="proportion").fillna(0)
wide["shift_a_to_c"] = wide.get("era_c", 0) - wide.get("era_a", 0)
wide.sort_values("shift_a_to_c").to_csv(OUT_DIR / "topic_by_era_wide.csv")

yr_tot = clean.groupby("year").size()
by_year = clean.groupby(["year", "topic"]).size().reset_index(name="chunks")
by_year["proportion"] = by_year.apply(lambda r: r.chunks / yr_tot[r.year], axis=1)
by_year.to_csv(OUT_DIR / "topic_by_year.csv", index=False)

tot = topic_model.topics_over_time(fit_texts, timestamps=fit_df["year"].tolist(), nr_bins=12, evolution_tuning=True)
tot.to_csv(OUT_DIR / "topics_over_time_bertopic.csv", index=False)

top_env = env_df[env_df.topic != -1].head(6).topic.tolist()
fig, ax = plt.subplots(figsize=(10, 4.5))
for t in top_env:
    g = by_year[by_year.topic == t]
    label = ", ".join(terms_df[terms_df.topic == t].term.head(4))
    ax.plot(g.year, g.proportion, marker="o", ms=3, label=f"{t}: {label}")
for x, name in ((1962, "Silent Spring"), (1972, "Clean Water Act")):
    ax.axvline(x, color="grey", ls="--", alpha=0.5); ax.text(x, ax.get_ylim()[1], name, fontsize=8, ha="left", va="top")
ax.set_xlabel("year"); ax.set_ylabel("share of chunks"); ax.set_title("Top env-scored topics over time")
ax.legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left"); fig.tight_layout()
fig.savefig(OUT_DIR / "env_topics_over_time.png", dpi=150); plt.close(fig)
print(wide.sort_values("shift_a_to_c").tail(8))

## 12. Volume level — topic proportions and mean-pooled vectors


In [ ]:
vol_topic = (clean.groupby(["htid", "topic"]).size().unstack(fill_value=0))
vol_topic = vol_topic.div(vol_topic.sum(axis=1), axis=0)
vol_topic.to_csv(OUT_DIR / "volume_topic_proportions.csv")

order = [h for h in volumes if vec_n[h] > 0]
V = np.vstack([vec_sum[h] / vec_n[h] for h in order]).astype(np.float16)   # 2,881 x 384 = 2.2 MB at float16, 4.4 at float32
np.save(OUT_DIR / "volume_vectors.npy", V)
# Over the 1 MB review threshold at 2,881 volumes: name it in the release request or split by era.
pd.DataFrame({"htid": order, "n_chunks": [vec_n[h] for h in order],
              "era": [meta_lookup[h]["era"] for h in order], "year": [meta_lookup[h][YEAR_COL] for h in order],
              "source": [meta_lookup[h][SOURCE_COL] for h in order],
              "title": [meta_lookup[h].get("title", "") for h in order]}).to_csv(OUT_DIR / "volume_vectors_index.csv", index=False)
print(vol_topic.shape, V.shape)

## 13. Stability across seeds


In [ ]:
def topic_words(model):
    return {t: {w for w, _ in (model.get_topic(t) or [])} for t in set(model.topics_) if t != -1}

base_words = topic_words(topic_model)
stab_rows = []
for seed in SEEDS[1:]:
    m = make_model(seed=seed)
    m.fit_transform(fit_texts, fit_emb)
    other = topic_words(m)
    for t, ws in base_words.items():
        best = max((len(ws & ow) / len(ws | ow) for ow in other.values()), default=0.0)
        stab_rows.append({"topic": t, "seed": seed, "best_jaccard": round(best, 4), "n_topics_refit": len(other)})
    print(f"seed {seed}: {len(other)} topics")
if stab_rows:
    stab = pd.DataFrame(stab_rows)
    summary = stab.groupby("topic").best_jaccard.mean().rename("mean_best_jaccard").reset_index()
    summary.to_csv(OUT_DIR / "topic_stability.csv", index=False)
    print(summary.sort_values("mean_best_jaccard").head(10).to_string(index=False))

## 14. Manifest and export check

Release with one `releaseresults add` of every listed file, then `releaseresults done`. Repeated adds overwrite each other.


In [ ]:
manifest = {
    "embedder": str(MODEL_PATH), "token_budget": TOKEN_BUDGET,
    "chunk_words": CHUNK_WORDS, "overlap_words": OVERLAP_WORDS, "chunks_split_past_budget": enforce_budget.oversized,
    "year_cutoffs": YEAR_CUTOFFS, "volumes": len(volumes), "chunks_total": int(len(chunk_df)),
    "chunks_fit": int(len(fit_df)), "sample_per_vol": SAMPLE_PER_VOL, "balance_eras": BALANCE_ERAS,
    "min_cluster_size": HDBSCAN_MIN_CLUSTER, "min_df": MIN_DF, "topics_raw": n_topics_raw,
    "topics_final": int(len(set(topics)) - (1 if -1 in topics else 0)),
    "outlier_share_all": round(float((all_df.topic == -1).mean()), 4),
    "seeds": SEEDS, "runtime_sec_since_chunking": round(time.time() - t0, 1),
    "clean_totals": {k: int(sum(s[k] for s in clean_stats.values())) for k in ("pages", "dropped_lines", "dropped_pages", "dehyphenated")},
}
json.dump(manifest, (OUT_DIR / "run_manifest.json").open("w"), indent=2)

total = 0
for p in sorted(OUT_DIR.iterdir()):
    sz = p.stat().st_size; total += sz
    print(f"  {p.name:40s} {sz/1e3:8.1f} KB")
print(f"export dir total: {total/1e6:.2f} MB" + ("  ⚠ over HTRC's 1 MB threshold — trim before requesting release" if total > 1_048_576 else ""))
print("\nrelease with ONE add (repeated adds OVERWRITE - each wipes the release spool):")
print("  releaseresults add " + " ".join(str(p) for p in sorted(OUT_DIR.iterdir())))
print("  releaseresults done")